# FOLTRA Week 5 on Colab GPU

On your PC, from Foltra, run `python -B -m scripts.colab.package`.
Upload the two ZIPs from `experiments/week_05_baseline/outputs/colab_upload/`
to a Google Drive folder named `FOLTRA` under My Drive. Only PlantVillage Color is needed.

Select **Runtime > Change runtime type > GPU** (T4 if offered).
Run cells in order. Full training requires explicitly setting RUN_FULL_TRAINING=True.
Code and results live on Drive; images are extracted to the runtime's local disk.
Existing manifests are unchanged. This notebook creates a separate Colab configuration.
Colab runtimes can expire; best checkpoints are saved directly to Drive each time
validation loss improves. Training currently starts fresh and does not resume checkpoints.
[Colab FAQ](https://research.google.com/colaboratory/faq.html).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import shutil, zipfile, os, sys, subprocess, uuid
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime, then rerun this notebook.')
print('GPU:', torch.cuda.get_device_name(0))
DRIVE_FOLDER = Path('/content/drive/MyDrive/FOLTRA')
for name in ('foltra_code.zip', 'plantvillage_color.zip'):
    if not (DRIVE_FOLDER / name).is_file():
        raise FileNotFoundError(DRIVE_FOLDER / name)


def run_live(command):
    """Forward child output through the notebook's own output stream."""
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
        if status:
            raise subprocess.CalledProcessError(status, command)
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()


In [ ]:
# Use a fresh code directory on Drive so previous runs are never overwritten.
PROJECT = DRIVE_FOLDER / ('project_' + uuid.uuid4().hex[:8])
PROJECT.mkdir()
print('Extracting code...', flush=True)
with zipfile.ZipFile(DRIVE_FOLDER / 'foltra_code.zip') as archive:
    archive.extractall(PROJECT)
# Copy the archive once, then read training images from local storage.
LOCAL_DATA = Path('/content/foltra_data_' + uuid.uuid4().hex[:8])
LOCAL_DATA.mkdir()
local_zip = LOCAL_DATA / 'plantvillage_color.zip'
print('Copying dataset ZIP to local storage...', flush=True)
shutil.copy2(DRIVE_FOLDER / 'plantvillage_color.zip', local_zip)
print('Extracting dataset images...', flush=True)
with zipfile.ZipFile(local_zip) as archive:
    archive.extractall(LOCAL_DATA)
local_zip.unlink()
os.chdir(PROJECT)
run_live([sys.executable, '-u', '-m', 'pip', 'install', '-r', 'requirements.txt'])
print('Project and persistent outputs:', PROJECT)


In [ ]:
import yaml, csv
# Modify only the freshly extracted upload, not your PC repository.
path = PROJECT / 'configs/datasets.yaml'
cfg = yaml.safe_load(path.read_text())
cfg['paths']['dataset_root'] = str(LOCAL_DATA / 'Datasets')
path.write_text(yaml.safe_dump(cfg, sort_keys=False))
config = yaml.safe_load((PROJECT / 'configs/baseline.yaml').read_text())
config['device'] = 'cuda'
config['training']['num_workers'] = 2
(PROJECT / 'configs/baseline_colab.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
# Validate every referenced file before spending GPU time training.
for split, name in config['manifests'].items():
    with (PROJECT / 'data/manifests' / name).open(newline='', encoding='utf-8-sig') as handle:
        rows = list(csv.DictReader(handle))
    missing = [row['path'] for row in rows if not (LOCAL_DATA / row['path']).is_file()]
    if missing:
        raise FileNotFoundError(f'{split}: {len(missing)} missing images; first: {missing[0]}')
    print(split, len(rows), 'files present')


Run the smoke test first. Confirm the output says `Device: cuda`. Small smoke scores do not measure model quality.

In [ ]:
run_live([sys.executable, '-u', '-B', '-m', 'src.training.train_baseline', '--config', 'baseline_colab', '--smoke'])


Set the flag below to True when ready for full training. Results and best checkpoints are saved directly to the project folder on Drive. Avoid running this cell twice concurrently.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    run_live([sys.executable, '-u', '-B', '-m', 'src.training.train_baseline', '--config', 'baseline_colab'])
else:
    print('Full training not started. Set RUN_FULL_TRAINING = True when ready.')
print('Results:', PROJECT / 'experiments/week_05_baseline/outputs')
